### 🛰️ Step X: Download Landsat ARD Surface Reflectance Tiles via STAC API

In this step, we automate the retrieval of **U.S. Landsat Collection 2 Analysis Ready Data (ARD)** — specifically the **Surface Reflectance (SR)** product — using the **USGS LandsatLook STAC API**.

Each query targets the `landsat-c2ard-sr` collection and filters tiles by spatial index, date, and cloud cover to assemble a clean, analysis-ready dataset for the study region.

We will:

1. **Query the STAC API**  
   - Endpoint: `https://landsatlook.usgs.gov/stac-server`  
   - Collection: `landsat-c2ard-sr` (U.S. ARD Surface Reflectance)  
   - Retrieves all ARD tiles matching the specified horizontal and vertical indices.

2. **Apply selection filters**  
   - Horizontal tiles: **H = 17–19**  
   - Vertical tiles: **V = 7–8**  
   - **Cloud cover < 10 %**  
   - Target months and years:  
     - **July (2022–2025)**  
     - **August (2010–2025)**

3. **Handle unstable API responses**  
   - Divides each month into daily date windows.  
   - Implements retry and exponential backoff to recover from 5xx “Internal Server Error” responses.

4. **Download ARD Surface Reflectance assets**  
   - Includes all bands (`SR_B1–SR_B7`) and QA layers (`SR_QA_PIXEL`, etc.).  
   - Saves each tile’s STAC metadata (`.stac.json`) for provenance.

5. **Organize downloaded files**  
   Files are automatically written to:

    /Users/kbv5173/Library/CloudStorage/GoogleDrive-vanmeterlab@gmail.com
    /
    My Drive/Research/Projects/NASA - UMRB Legacy Wetlands/Landsat/Iowa/{year}/{year}_{month}_Landsat_ARD/H###_V###/


6. **Log progress**  
For each month and year, the script prints the number of matched tiles and downloaded assets, showing per-file progress bars during transfer.

---

**Dependencies:**  
`pystac-client`, `requests`, `tqdm`

**Purpose:**  
Generate a reproducible, automated pipeline for collecting **cloud-filtered, tile-based Landsat ARD SR data** across multiple years for NDVI compositing and long-term time-series analysis.


In [2]:
#!/usr/bin/env python3
# Landsat C2 L2 (scene-level) downloader — RESUMABLE + MANIFEST + VERIFY
# - Queries MPC 'landsat-c2-l2' by bbox/date/cloud/platform
# - Downloads red, nir08 (+ QA with fallbacks) in parallel
# - Skips valid existing files, writes per-month manifest, verify at end
# - Robust re-sign on 401/403 and retries on transient errors
# - Blocks non-TIFF payloads (HTML/JSON/XML), peeks TIFF header before writing

from pathlib import Path
from datetime import datetime
from calendar import monthrange
from concurrent.futures import ThreadPoolExecutor, as_completed
import json, time

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm
from pystac_client import Client
import planetary_computer as pc
import os

# ========================
# USER SETTINGS
# ========================
OUTDIR         = Path(os.environ.get("DML_NDVI_DATA_ROOT", "./ndvi_wetlands_data")) / "Landsat/Iowa"
YEARS          = [2018]
MONTHS         = [9]                    # July, August
BBOX_WGS84     = [-96.7, 40.2, -90.0, 43.7]
CLOUD_MAX      = 60
PLATFORMS      = ["landsat-5","landsat-7","landsat-8","landsat-9"]

INCLUDE_QA     = True
PAGE_LIMIT     = 1000

# Networking / performance
TIMEOUT        = 120
CHUNK_BYTES    = 8 * 1024 * 1024
MAX_WORKERS    = 8
MAX_ATTEMPTS   = 5
BASE_SLEEP     = 1.0

# ========================
# HTTP session (pooling)
# ========================
SESSION = requests.Session()
adapter = HTTPAdapter(
    pool_connections=64,
    pool_maxsize=64,
    max_retries=Retry(total=0),  # we'll handle retries ourselves
)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)

# ========================
# Helpers
# ========================
def ym_date_range(y, m):
    last = monthrange(y, m)[1]
    return f"{datetime(y, m, 1).date().isoformat()}/{datetime(y, m, last).date().isoformat()}"

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def ok_tiff(path: Path, expect_large: bool = True) -> bool:
    """Validate TIFF by magic bytes; enforce size floor only for red/nir08."""
    try:
        with open(path, "rb") as f:
            magic = f.read(4)
        if magic not in (b"II*\x00", b"MM\x00*"):
            return False
        if expect_large and path.stat().st_size < 4_000_000:  # ~4 MB floor for spectral bands
            return False
        return True
    except Exception:
        return False

def save_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w") as f:
        json.dump(obj, f, indent=2)

def load_json(p: Path):
    try:
        with open(p, "r") as f:
            return json.load(f)
    except Exception:
        return None

# QA fallback map across sensors/years
PRIMARY_KEYS = ["red", "nir08"]
QA_MAP = {
    "qa_pixel": ["qa_pixel", "cloud_qa", "qa"],  # normalize to qa_pixel filename
    "qa_radsat": ["qa_radsat"],
}

def first_present(assets, candidates):
    for k in candidates:
        a = assets.get(k)
        if a and getattr(a, "href", None):
            return k
    return None

def stream_download(url: str, dest: Path, label: str, position: int):
    """Stream download one asset; reject non-TIFF responses and peek TIFF header first."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    with SESSION.get(url, stream=True, timeout=TIMEOUT, allow_redirects=True) as r:
        r.raise_for_status()
        ctype = (r.headers.get("Content-Type") or "").lower()
        if any(x in ctype for x in ("html", "xml", "json", "text/")):
            raise requests.HTTPError(f"Unexpected content-type {ctype}", response=r)

        # Peek first 4 bytes to validate TIFF magic before writing body
        r.raw.decode_content = True  # ensure compressed responses are decoded
        first = r.raw.read(4)
        if first not in (b"II*\x00", b"MM\x00*"):
            raise requests.HTTPError("Missing TIFF magic header", response=r)

        total = r.headers.get("Content-Length")
        total = int(total) if (total and total.isdigit()) else None
        show_bar = True if (total is None or total >= 20_000_000) else False
        pbar = tqdm(total=total, unit="B", unit_scale=True, desc=label,
                    position=position, leave=False) if show_bar else None

        with open(tmp, "wb") as f:
            f.write(first)
            if pbar and total: pbar.update(len(first))
            for chunk in r.iter_content(CHUNK_BYTES):
                if chunk:
                    f.write(chunk)
                    if pbar and total:
                        pbar.update(len(chunk))
        if pbar: pbar.close()

    tmp.replace(dest)

def resilient_fetch(item, asset_key: str, dest: Path, label: str, position: int):
    """Download with retries; re-sign each attempt; treat QA as optional on final failure."""
    attempt = 0
    while True:
        attempt += 1
        try:
            signed = pc.sign(item)  # re-sign every attempt (refresh SAS)
            asset = signed.assets.get(asset_key)
            if not asset or not asset.href:
                raise RuntimeError(f"Missing asset '{asset_key}' for {item.id}")
            stream_download(asset.href, dest, label, position)

            # Post-check: red/nir08 should be "large"; QA can be small
            expect_large = asset_key in ("red", "nir08")
            if not ok_tiff(dest, expect_large=expect_large):
                dest.unlink(missing_ok=True)
                raise RuntimeError("Downloaded file failed TIFF validation")
            return

        except requests.HTTPError as e:
            status = getattr(e.response, "status_code", None)
            if status in (401, 403, 429, 500, 502, 503, 504) and attempt < MAX_ATTEMPTS:
                time.sleep(BASE_SLEEP * (2 ** (attempt - 1)))
                continue
            # After last attempt: skip QA, raise for spectral
            if asset_key in ("qa_pixel","cloud_qa","qa","qa_radsat"):
                print(f"[INFO] Skipping optional QA after retries: {item.id}:{asset_key} ({e})")
                return
            raise
        except Exception as e:
            if attempt < MAX_ATTEMPTS:
                time.sleep(BASE_SLEEP * (2 ** (attempt - 1)))
                continue
            if asset_key in ("qa_pixel","cloud_qa","qa","qa_radsat"):
                print(f"[INFO] Skipping optional QA after retries: {item.id}:{asset_key} ({e})")
                return
            raise

def download_all(plan):
    """Run batch downloads in parallel. plan: (item, asset_key, dest, label, position)."""
    if not plan:
        return
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {
            ex.submit(resilient_fetch, it, key, dest, label, pos): (dest, key, it.id)
            for (it, key, dest, label, pos) in plan
        }
        for fut in as_completed(futures):
            dest, key, sid = futures[fut]
            try:
                fut.result()
            except Exception as e:
                print(f"[WARN] {sid}:{dest.name} ({key}) → {e}")

# ========================
# Main
# ========================
def main():
    CAT = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

    for y in YEARS:
        for m in MONTHS:
            dr = ym_date_range(y, m)
            month_root = ensure_dir(OUTDIR / f"{y}" / f"{y}_{m:02d}_Landsat_SCENES")
            manifest_path = month_root / "_manifest.json"

            print(f"[SEARCH] {y}-{m:02d}  dates: {dr}")
            try:
                s = CAT.search(
                    collections=["landsat-c2-l2"],
                    datetime=dr,
                    bbox=BBOX_WGS84,
                    query={"eo:cloud_cover": {"lte": CLOUD_MAX}, "platform": {"in": PLATFORMS}},
                    limit=PAGE_LIMIT,
                )
                items = list(s.get_all_items())
            except Exception as e:
                print(f"[WARN] STAC search failed for {y}-{m:02d}: {e}")
                continue

            print(f"  found scene items: {len(items)}")

            # Build manifest of expected files for this month
            manifest = {"year": y, "month": m, "items": {}}
            for item in items:
                sid = item.id
                out_dir = ensure_dir(month_root / sid)
                signed = pc.sign(item)  # cheap; reading keys only
                assets = signed.assets

                want = []
                # primary bands
                for key in PRIMARY_KEYS:
                    if key in assets and getattr(assets[key], "href", None):
                        want.append((key, f"{sid}_{key}.TIF"))

                # QA with fallbacks
                if INCLUDE_QA:
                    k_pix = first_present(assets, QA_MAP["qa_pixel"])
                    if k_pix:
                        want.append((k_pix, f"{sid}_qa_pixel.TIF"))   # normalized filename
                    k_rad = first_present(assets, QA_MAP["qa_radsat"])
                    if k_rad:
                        want.append((k_rad, f"{sid}_qa_radsat.TIF"))

                manifest["items"][sid] = [fname for (_k, fname) in want]

                # Save tiny provenance STAC JSON once
                stac_p = out_dir / f"{sid}.stac.json"
                if not stac_p.exists():
                    try:
                        save_json(stac_p, item.to_dict())
                    except Exception:
                        pass

            save_json(manifest_path, manifest)

            # Plan downloads ONLY for missing/invalid files
            month_plan = []
            bar_slot = 0
            missing_total = 0

            # Quick lookup: id -> item
            lookup = {it.id: it for it in items}

            for sid, file_list in manifest["items"].items():
                out_dir = month_root / sid
                it = lookup.get(sid)
                if it is None:
                    continue

                signed = pc.sign(it)
                assets = signed.assets

                def resolve_key(norm_name: str):
                    if norm_name.endswith("_red.TIF"): return "red"
                    if norm_name.endswith("_nir08.TIF"): return "nir08"
                    if norm_name.endswith("_qa_radsat.TIF"):
                        return "qa_radsat" if "qa_radsat" in assets else None
                    if norm_name.endswith("_qa_pixel.TIF"):
                        return first_present(assets, QA_MAP["qa_pixel"])
                    return None

                for fname in file_list:
                    dest = out_dir / fname
                    # Expect large for spectral, relaxed for QA
                    expect_large = fname.endswith(("_red.TIF", "_nir08.TIF"))
                    if dest.exists() and ok_tiff(dest, expect_large=expect_large):
                        continue
                    key = resolve_key(fname)
                    if not key or key not in assets or not getattr(assets[key], "href", None):
                        # can't resolve this asset key; leave for next run/log
                        continue
                    month_plan.append((it, key, dest, dest.name, bar_slot % MAX_WORKERS))
                    bar_slot += 1
                    missing_total += 1

            print(f"  → queued missing/invalid files: {missing_total}")
            download_all(month_plan)

            # VERIFY PASS
            still_missing = []
            man = load_json(manifest_path) or manifest
            for sid, file_list in man["items"].items():
                for fname in file_list:
                    p = month_root / sid / fname
                    expect_large = fname.endswith(("_red.TIF", "_nir08.TIF"))
                    if not (p.exists() and ok_tiff(p, expect_large=expect_large)):
                        still_missing.append(str(p))

            if still_missing:
                print(f"[VERIFY] {y}-{m:02d}: {len(still_missing)} file(s) still missing/invalid (safe to rerun):")
                for p in still_missing[:12]:
                    print("   -", p)
                if len(still_missing) > 12:
                    print(f"   … (+{len(still_missing)-12} more)")
            else:
                print(f"[VERIFY] {y}-{m:02d}: all expected files present.")

    print("\n[DONE] Resume+verify run complete. Re-run anytime; it only fills gaps.")

if __name__ == "__main__":
    main()


[SEARCH] 2018-09  dates: 2018-09-01/2018-09-30


/opt/anaconda3/envs/DML_wetlands_111725/lib/python3.12/site-packages/pystac_client/item_search.py:940: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


  found scene items: 48
  → queued missing/invalid files: 177





_029029_20180928_02_T1_nir08.TIF:   0%|          | 0.00/98.5M [00:00<?, ?B/s]






0925_02_T1_red.TIF:   0%|          | 0.00/101M [00:00<?, ?B/s]

2SP_029029_20180928_02_T1_red.TIF:   0%|          | 0.00/92.1M [00:00<?, ?B/s]





0180925_02_T1_nir08.TIF:   0%|          | 0.00/104M [00:00<?, ?B/s]




2_20180925_02_T1_red.TIF:   0%|          | 0.00/102M [00:00<?, ?B/s]



3032_20180926_02_T1_red.TIF:   0%|          | 0.00/59.9M [00:00<?, ?B/s]

2SP_029029_20180928_02_T1_red.TIF:   9%|▉         | 8.39M/92.1M [01:59<19:52, 70.2kB/s]

2SP_029029_20180928_02_T1_red.TIF:   9%|▉         | 8.39M/92.1M [02:10<19:52, 70.2kB/s]


_029029_20180928_02_T1_nir08.TIF:   9%|▊         | 8.39M/98.5M [02:16<24:29, 61.3kB/s]



3032_20180926_02_T1_red.TIF:  14%|█▍        | 8.39M/59.9M [02:09<13:15, 64.8kB/s]




2_20180925_02_T1_red.TIF:   8%|▊         | 8.39M/102M [02:20<26:06, 59.7kB/s]

[WARN] LE07_L2SP_028031_20180929_02_T2:LE07_L2SP_028031_20180929_02_T2_red.TIF (red) → Downloaded file failed TIFF validation


...

[WARN] LE07_L2SP_028032_20180929_02_T2:LE07_L2SP_028032_20180929_02_T2_red.TIF (red) → Downloaded file failed TIFF validation



7_L2SP_025032_20180924_02_T1_red.TIF:   0%|          | 0.00/66.6M [00:00<?, ?B/s]



LC08_L2SP_024031_20180925_02_T1_nir08.TIF:   8%|▊         | 8.39M/104M [02:00<22:45, 69.9kB/s]

2SP_029029_20180928_02_T1_red.TIF:  18%|█▊        | 16.8M/92.1M [04:30<19:38, 63.9kB/s]




2_20180925_02_T1_red.TIF:  16%|█▋        | 16.8M/102M [04:23<20:47, 68.2kB/s]



LC08_L2SP_024031_20180925_02_T1_nir08.TIF:   8%|▊         | 8.39M/104M [02:10<22:45, 69.9kB/s]






0925_02_T1_red.TIF:  17%|█▋        | 16.8M/101M [04:49<23:55, 58.9kB/s]






0925_02_T1_red.TIF:  17%|█▋        | 16.8M/101M [05:00<23:55, 58.9kB/s]





0180925_02_T1_nir08.TIF:   8%|▊         | 8.39M/104M [05:41<1:04:30, 24.6kB/s]





0180925_02_T1_nir08.TIF:   8%|▊         | 8.39M/104M [05:53<1:04:30, 24.6kB/s]

2SP_029029_20180928_02_T1_red.TIF:  27%|██▋       | 25.2M/92.1M [06:28<17:21, 64.3kB/s]

2SP_029029_20180928_02_T1_red.TIF:  27%|██▋       | 25.2M/92.1M [06:40<17:21, 64.3kB/s]




2_20180925_02_T1_red.TIF:  25%|██▍       | 2

[WARN] LC08_L2SP_024031_20180925_02_T1:LC08_L2SP_024031_20180925_02_T1_red.TIF (red) → HTTPSConnectionPool(host='landsateuwest.blob.core.windows.net', port=443): Read timed out.
[WARN] LC08_L2SP_024032_20180925_02_T1:LC08_L2SP_024032_20180925_02_T1_nir08.TIF (nir08) → HTTPSConnectionPool(host='landsateuwest.blob.core.windows.net', port=443): Read timed out.


0180922_02_T1_nir08.TIF: 100%|██████████| 67.3M/67.3M [1:04:32<00:00, 13.2kB/s]

2SP_026029_20180923_02_T1_nir08.TIF: 100%|██████████| 99.0M/99.0M [1:07:48<00:00, 18.4kB/s]

                                                                                         




0_20180923_02_T1_red.TIF:  85%|████████▍ | 75.5M/88.8M [1:03:02<11:34, 19.2kB/s]
8_L2SP_026029_20180923_02_T1_red.TIF:  85%|████████▌ | 75.5M/88.3M [1:02:02<11:01, 19.4kB/s]





0180923_02_T1_nir08.TIF:  86%|████████▌ | 83.9M/97.8M [1:05:32<11:22, 20.4kB/s]




0_20180923_02_T1_red.TIF:  85%|████████▍ | 75.5M/88.8M [1:03:02<11:34, 19.2kB/s]
8_L2SP_026029_20180923_02_T1_red.TIF:  85%|████████▌ | 75.5M/88.3M [1:02:02<11:01, 19.4kB/s]





0180923_02_T1_nir08.TIF:  86%|████████▌ | 83.9M/97.8M [1:05:32<11:22, 20.4kB/s]




                                                                              





                                                                               

                                         

[VERIFY] 2018-09: 4 file(s) still missing/invalid (safe to rerun):
   - /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/Landsat/Iowa/2018/2018_09_Landsat_SCENES/LE07_L2SP_028032_20180929_02_T2/LE07_L2SP_028032_20180929_02_T2_red.TIF
   - /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/Landsat/Iowa/2018/2018_09_Landsat_SCENES/LE07_L2SP_028031_20180929_02_T2/LE07_L2SP_028031_20180929_02_T2_red.TIF
   - /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/Landsat/Iowa/2018/2018_09_Landsat_SCENES/LC08_L2SP_024032_20180925_02_T1/LC08_L2SP_024032_20180925_02_T1_nir08.TIF
   - /Volumes/Conowingo/NASA_UMRB_Legacy_Wetlands/Landsat/Iowa/2018/2018_09_Landsat_SCENES/LC08_L2SP_024031_20180925_02_T1/LC08_L2SP_024031_20180925_02_T1_red.TIF

[DONE] Resume+verify run complete. Re-run anytime; it only fills gaps.
